# Game-Conditioned Kandinsky — Step 11: Paired Data Contract

This notebook prepares a rights-aware paired-data contract for a narrow impact/collision pilot. It does **not** download media, train a model, or claim any sample is cleared for commercial training.

The pilot aligns five-second video and target audio with one time-stamped event. It follows a strict separation: measured mass and normal impact speed are physical generator inputs; measured impulse is an optional validation field; microphone distance and room reverberation are recorded separately for a later renderer, not mixed into the impact token.

## 1. Create a clean workspace

The generated JSONL file is a blank template. You will later fill it only with recordings or assets whose rights have been reviewed.

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
from datetime import datetime, timezone

WORK_DIR = Path("/content/game_audio_data_contract")
WORK_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = WORK_DIR / "paired_samples.jsonl"
print("Workspace:", WORK_DIR)
print("Manifest template:", MANIFEST_PATH)

## 2. Define the event record and keep physical controls separate

Use SI units and seconds relative to the five-second clip. Unknown values stay `null`; never replace missing measurements with zero or guesses. Preserve the raw engine event when available so engine-specific integrations can be audited. Lab-captured records may leave engine metadata empty.

For the first impact pilot, use material pair, measured moving-object mass, and measured normal approach speed as the primary event fields. Impulse is optional supporting evidence. Keep the recording dry with fixed microphone position and gain. Record distance and room/RT60 as capture metadata for later renderer tests; they are **not** part of the Game→Audio impact vector in this pilot.

Every video and audio asset needs its own rights evidence. A dataset-level license label alone does not clear underlying footage or sound for commercial model training.

In [ ]:
def asset_template(relative_path):
    return {
        "path": relative_path,
        "sha256": None,
        "duration_s": None,
        "measured_streams": [],
        "rights": {
            "rights_holder": None,
            "license_id": None,
            "license_text_url": None,
            "commercial_use_allowed": False,
            "commercial_model_training_allowed": False,
            "attribution_required": None,
            "attribution_text": None,
            "evidence_path_or_url": None,
            "review_status": "unverified"
        }
    }

def measured_value(value=None, provenance="unknown", method=None):
    return {"value": value, "provenance": provenance, "measurement_method": method}

TEMPLATE_RECORD = {
    "schema_version": "game-audio-pair-v0.2",
    "sample_id": "pilot_impact_0001",
    "capture_session_id": "replace_with_session_id",
    "split": "train",
    "clip_duration_s": 5.0,
    "text_prompt": "Audio that matches the provided gameplay video.",
    "prompt_policy": "fixed_generic_for_controlled_comparison",
    "video": asset_template("media/video/pilot_impact_0001.mp4"),
    "target_audio": asset_template("media/audio/pilot_impact_0001.wav"),
    "engine": {
        "name": None,
        "version": None,
        "game_or_scene_id": None,
        "raw_event_name": None,
        "raw_event_payload": None
    },
    "recording_conditions": {
        "microphone_model": None,
        "mic_position_relative_to_contact_xyz_m": None,
        "gain_db": None,
        "agc_enabled": None,
        "sample_rate_hz": None,
        "bit_depth": None,
        "raw_audio_is_dry": None,
        "room_profile": None,
        "measured_rt60_s": None
    },
    "events": [
        {
            "event_id": "event_001",
            "event_type": "impact",
            "event_time_s": 2.1,
            "audio_onset_s": None,
            "alignment_confidence": None,
            "physical_event": {
                "material_a": None,
                "material_b": None,
                "mass_a_kg": measured_value(),
                "normal_relative_speed_m_s": measured_value(),
                "impulse_ns": measured_value(),
                "bbox_xyz_m": measured_value(),
                "contact_position_local_m": None,
                "intensity_control_01": measured_value()
            }
        }
    ],
    "recorded_at_utc": datetime.now(timezone.utc).isoformat()
}

if MANIFEST_PATH.exists():
    print("Existing manifest preserved:", MANIFEST_PATH)
else:
    MANIFEST_PATH.write_text(
        json.dumps(TEMPLATE_RECORD, ensure_ascii=False) + "\n",
        encoding="utf-8"
    )
    print("Created blank template:", MANIFEST_PATH)
print(json.dumps(TEMPLATE_RECORD, ensure_ascii=False, indent=2))

In [ ]:
import subprocess

def inspect_media_file(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    if not shutil.which("ffprobe"):
        raise RuntimeError("ffprobe is unavailable; use a Colab runtime with FFmpeg installed.")
    result = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries",
        "format=duration:stream=codec_type,codec_name,sample_rate,channels,width,height,avg_frame_rate",
        "-of", "json", str(path)
    ], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-2000:])
    probe = json.loads(result.stdout)
    digest = hashlib.sha256()
    with path.open("rb") as media_file:
        for chunk in iter(lambda: media_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return {
        "sha256": digest.hexdigest(),
        "duration_s": float(probe["format"]["duration"]),
        "measured_streams": probe.get("streams", [])
    }

def refresh_asset_metadata(record, asset_name):
    asset = record[asset_name]
    measured = inspect_media_file(WORK_DIR / asset["path"])
    asset.update(measured)
    return measured

print("Media inspector ready. Run it after the paired video and WAV have been placed under WORK_DIR.")

## 3. Check rights, measured media, and paired event timing

The validator is deliberately conservative. It checks the declared files, measured durations and hashes, event times, and required rights evidence. It cannot decide whether a license legally permits commercial model training; a human must review the actual terms and evidence. The blank template is expected to fail.

In [ ]:
def validate_record(record, root=WORK_DIR, require_files=True):
    errors = []
    if record.get("schema_version") != "game-audio-pair-v0.2":
        errors.append("Unsupported or missing schema_version")
    if record.get("clip_duration_s") != 5.0:
        errors.append("This pilot expects five-second paired clips")
    if not record.get("capture_session_id") or record["capture_session_id"].startswith("replace_"):
        errors.append("Set capture_session_id; split by session/specimen to avoid leakage")
    if not record.get("text_prompt") or record.get("prompt_policy") != "fixed_generic_for_controlled_comparison":
        errors.append("Use and record one fixed generic prompt for controlled comparisons")

    for asset_name in ("video", "target_audio"):
        asset = record.get(asset_name) or {}
        path = asset.get("path")
        if not path:
            errors.append(f"{asset_name}: missing path")
        elif require_files and not (root / path).is_file():
            errors.append(f"{asset_name}: file not found: {path}")
        duration = asset.get("duration_s")
        if duration is None or abs(duration - 5.0) > 0.05:
            errors.append(f"{asset_name}: measured duration must be within 50 ms of five seconds")
        if not asset.get("sha256"):
            errors.append(f"{asset_name}: calculate and record a SHA-256 hash")
        if not asset.get("measured_streams"):
            errors.append(f"{asset_name}: inspect and record stream metadata")
        rights = asset.get("rights") or {}
        for key in ("rights_holder", "license_id", "license_text_url", "evidence_path_or_url"):
            if not rights.get(key):
                errors.append(f"{asset_name}: missing rights.{key}")
        if rights.get("commercial_use_allowed") is not True:
            errors.append(f"{asset_name}: commercial use is not verified as allowed")
        if rights.get("commercial_model_training_allowed") is not True:
            errors.append(f"{asset_name}: commercial model training is not verified as allowed")
        if rights.get("review_status") != "verified":
            errors.append(f"{asset_name}: rights.review_status must be 'verified'")

    video_duration = (record.get("video") or {}).get("duration_s")
    audio_duration = (record.get("target_audio") or {}).get("duration_s")
    if video_duration is not None and audio_duration is not None and abs(video_duration - audio_duration) > 0.05:
        errors.append("Video and target audio durations differ by more than 50 ms")

    conditions = record.get("recording_conditions") or {}
    if conditions.get("agc_enabled") is not False:
        errors.append("Disable and record AGC as false for the dry-source pilot")
    if conditions.get("gain_db") is None:
        errors.append("Record microphone gain in dB")
    if conditions.get("mic_position_relative_to_contact_xyz_m") is None:
        errors.append("Record the microphone position relative to the contact point")
    if conditions.get("raw_audio_is_dry") is not True:
        errors.append("The pilot source audio must be confirmed dry")

    events = record.get("events") or []
    if not events:
        errors.append("At least one event is required")
    for index, event in enumerate(events):
        t = event.get("event_time_s")
        if t is None or not (0.0 <= t < 5.0):
            errors.append(f"events[{index}]: event_time_s must be inside [0, 5)")
        onset = event.get("audio_onset_s")
        if onset is None or not (0.0 <= onset < 5.0):
            errors.append(f"events[{index}]: measure audio_onset_s inside [0, 5)")
        confidence = event.get("alignment_confidence")
        if confidence is None or not (0.0 <= confidence <= 1.0):
            errors.append(f"events[{index}]: alignment_confidence must be reviewed and in [0, 1]")
        if event.get("event_type") != "impact":
            errors.append(f"events[{index}]: Step 11 is scoped to impact events")
        physical = event.get("physical_event") or {}
        if not physical.get("material_a") or not physical.get("material_b"):
            errors.append(f"events[{index}]: record both contact materials")
        for key, minimum, allowed_provenance in (
            ("mass_a_kg", 0.0, {"measured", "engine_reported"}),
            ("normal_relative_speed_m_s", 0.0, {"measured", "engine_reported"}),
        ):
            item = physical.get(key) or {}
            value = item.get("value")
            if value is None or value <= minimum:
                errors.append(f"events[{index}].physical_event.{key}: provide a positive value")
            if item.get("provenance") not in allowed_provenance or not item.get("measurement_method"):
                errors.append(f"events[{index}].physical_event.{key}: record measured/engine-reported provenance and method")
    return errors

errors = validate_record(TEMPLATE_RECORD, require_files=False)
print("Template status:", "READY" if not errors else "NOT TRAINING READY")
print("Expected until real media, measurements, and rights evidence are added:")
for error in errors:
    print(" -", error)

## 4. First-pilot rules

- Keep the pilot to one event family: a single impact/collision between a moving object and a static surface.
- For each pair, video, dry target audio, and event telemetry must come from the same physical event and share a five-second timeline.
- Keep the text prompt fixed and generic; do not put material, mass, speed, or intensity in it.
- Record `mass_a_kg` and `normal_relative_speed_m_s` as measured or engine-reported values with a method/provenance. Leave unknown values null. Keep optional impulse as supporting evidence, not a second independent control.
- Record the raw engine event payload when available. Each engine adapter can map it to this canonical event record without pretending that every engine exposes identical physics fields.
- Capture a dry source at fixed microphone position/gain with AGC disabled. Record room and microphone metadata. Evaluate distance and reverb later in a separate renderer sweep.
- Hash every asset and retain rights evidence per file. Do not treat public availability or a top-level dataset license as sufficient proof of commercial model-training rights.
- This pilot is for controlled data/evaluation and does not guarantee that 180 events are enough to train a general model.

## What comes after this notebook

The local T01 plan already specifies a controlled wood-on-concrete impact matrix and written-rights requirements for commissioned capture. The next practical action is to use the same schema to get a feasibility quote and create a small rights-cleared paired capture. After the files exist, this notebook can inspect their measured metadata and alignment. Only after the pair set passes the audit should we decide whether a small adapter-training run is justified. The blank template is not training data.